# 10 — Differential Pair with Stacked Current Mirror Bias

**Dataset:** `datasets/diff_pair_stackedcmirror/`

Topology: 4 NMOS stacked cascode mirror (tail) + 4 NMOS diff pair (ABBA)
- Cascode mirror: M_rb(diode-bot) + M_rt(diode-top) + M_cb(copy-bot) + M_ct(copy-top) → VTAIL
- Diff pair (×4 ABBA): G=VP/VN, D=VDD1/VDD2, S=VTAIL

Nodes: `VP VN VDD1 VDD2 IBIAS_BOT IBIAS_TOP VSS`

In [ ]:
import sys, os
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
sys.path.insert(0, os.path.abspath('../../src/gelochip'))
import gelochip.gl as gl
gl.reload()  # pick up latest code without restarting kernel


In [ ]:
vp       = gl.Net('vp')
vn       = gl.Net('vn')
vdd1     = gl.Net('vdd1')
vdd2     = gl.Net('vdd2')
ib_bot   = gl.Net('ib_bot')   # bottom cascode bias (diode ref)
ib_top   = gl.Net('ib_top')   # top cascode bias (diode ref)
vmid_ref = gl.Net('vmid_ref') # ref stack mid
vmid_cpy = gl.Net('vmid_cpy') # copy stack mid
vtail    = gl.Net('vtail')    # copy stack output = diff pair tail

# Stacked cascode current mirror (4 NMOS)
m_rb = gl.nmos(w=4.0, fingers=1, g=ib_bot, d=ib_bot,   s=gl.gnd)  # bot ref diode
m_rt = gl.nmos(w=4.0, fingers=1, g=ib_top, d=ib_top,   s=ib_bot)  # top ref diode
m_cb = gl.nmos(w=4.0, fingers=1, g=ib_bot, d=vmid_cpy, s=gl.gnd)  # bot copy
m_ct = gl.nmos(w=4.0, fingers=1, g=ib_top, d=vtail,    s=vmid_cpy) # top copy → tail

# 4-NMOS diff pair ABBA
m_tl = gl.nmos(w=3.0, fingers=2, g=vp, d=vdd1, s=vtail)
m_bl = gl.nmos(w=3.0, fingers=2, g=vp, d=vdd1, s=vtail)
m_tr = gl.nmos(w=3.0, fingers=2, g=vn, d=vdd2, s=vtail)
m_br = gl.nmos(w=3.0, fingers=2, g=vn, d=vdd2, s=vtail)

chip = gl.build(m_rb, m_rt, m_cb, m_ct, m_tl, m_bl, m_tr, m_br,
                name='diff_pair_stackedcmirror')
chip.show()
chip.drc()
chip.sim()